# TP-Q27-THROUGHPUT-PROBE v1.0 — engine-throughput ladder for Qwen3.8-27B-FP8 on Kaggle 4×L4

**Question.** The Q1 / PIV runs served the 27B at ~50 generated tok/s aggregate (0.16 s per token per sequence,
`enforce_eager=True`, `max_num_seqs=16`, no prefix caching). Top ARC-AGI-2 entries run 27B-class models with
program-synthesis harnesses inside the same 12 h, which is only possible at several hundred tok/s.
**How fast can this checkpoint be served on this hardware without changing what the model is asked?**

**Design.** One subprocess per engine configuration (the notebook process never imports torch/vLLM, so every
configuration starts on empty GPUs). Each configuration measures, on real ARC evaluation prompts rendered through
the real Qwen chat template:

| phase | what | why |
|---|---|---|
| `decode@n` | n concurrent sequences × `OUTPUT_TOKENS` tokens, `ignore_eos=True` | capacity: aggregate and per-sequence tok/s (every sequence runs the full length) |
| `prefill_cold` / `prefill_warm` | 32 prompts, `max_tokens=1`, sent twice | prefill tok/s; warm/cold ratio shows whether prefix caching works on this hybrid checkpoint |
| `fingerprint` | 4 prompts, greedy, 64 tokens | token-level agreement vs the frozen engine — how far kernel numerics drift |

vLLM's own log lines (KV-cache size, max concurrency, graph-capture time) are parsed from each worker log;
`nvidia-smi` is sampled every 5 s while a configuration runs (peak memory per GPU).

**Ladder** (`CONFIG_LADDER`, Cell 1): `C0` frozen Q1 engine (anchor) → `C1` CUDA graphs → `C2` + custom all-reduce →
`C3` `max_num_seqs=64` → `C4` 128 → `C5` prefix caching → `C6` GDN `mamba_cache_mode="all"` (optional) → `C7` FP8 KV cache (optional).
A timed-out configuration is killed as a whole process group; a failed one is recorded with its traceback and the
ladder continues. Before each configuration the remaining budget is checked against a projection built from the
previous measurements; optional rows are skipped first.

**Profiles.** `SELFTEST_CPU` (fake engine, PC or Kaggle CPU, ~1 min; includes two synthetic rows that must fail and
time out) → `SMOKE_GPU` (C0 + C1, 128 tokens, ~20 min) → `PROBE_GPU` (full ladder, 1,024 tokens, ~1.5–2.5 h).

**Decision.** `analysis/throughput_verdict.json` → `best_config`, `recommended_engine_kwargs`, `target_met` (≥ 300 tok/s
aggregate) and a projection table (hours for K samples/task × 120/240 tasks). The recommended kwargs go into the
next SMOKE/PIV notebook's `ENGINE_CONFIG`; nothing else changes.


In [ ]:
# =====================================================================================
# CELL 1 — CONFIG: TP-Q27-THROUGHPUT-PROBE v1.0
# Engine-throughput ladder for Qwen3.8-27B-FP8 on Kaggle 4xL4 (vLLM 0.19.0, TP=4).
# Each engine configuration runs in its OWN subprocess (fresh CUDA state); a failed or
# hung configuration is recorded and the ladder continues. Nothing here changes what the
# model is asked to do -- only how fast the engine serves it.
# =====================================================================================
import os, sys, json, time, math, hashlib, shutil, subprocess, importlib.metadata, platform, signal, re
from pathlib import Path
from datetime import datetime, timezone

EXPERIMENT_ID = "TP-Q27-THROUGHPUT-PROBE-v1.0"
RUN_PROFILE = "PROBE_GPU"            # "SELFTEST_CPU" | "SMOKE_GPU" | "PROBE_GPU"
SEED = 42

# Optional: restrict the ladder to a subset of config names (None = profile default).
PROBE_CONFIGS = None                 # e.g. ["C0_frozen_q1", "C1_cudagraph"]

PROFILE_TABLE = {
    # Fake engine, no vLLM/CUDA. Runs on a PC or a Kaggle CPU session. Exercises the whole
    # orchestration: subprocess launch, timeout/kill, failure capture, parsing, analysis, bundle.
    # (The fake engine reports realistic, time-scaled load/decode numbers, so the deadline projection is fed
    # PROBE-like values; the budget is therefore the PROBE budget even though the wall clock is ~1 min.)
    "SELFTEST_CPU": {"use_model": False, "fake_engine": True,  "output_tokens": 64,   "prompt_pool": 16,
                     "config_filter": None, "per_config_timeout_sec": 120, "total_budget_sec": 5 * 60 * 60,
                     "bundle_name": "TP_Q27_PROBE_SELFTEST_BUNDLE.zip"},
    # Real model, 2 configs, short outputs: proves the real subprocess/vLLM path (~20 min GPU).
    "SMOKE_GPU":    {"use_model": True,  "fake_engine": False, "output_tokens": 128,  "prompt_pool": 32,
                     "config_filter": ["C0_frozen_q1", "C1_cudagraph"], "per_config_timeout_sec": 25 * 60,
                     "total_budget_sec": 90 * 60, "bundle_name": "TP_Q27_PROBE_SMOKE_BUNDLE.zip"},
    # Full ladder, 1,024-token outputs (~1.5-2.5 h GPU).
    "PROBE_GPU":    {"use_model": True,  "fake_engine": False, "output_tokens": 1024, "prompt_pool": 128,
                     "config_filter": None, "per_config_timeout_sec": 45 * 60, "total_budget_sec": 5 * 60 * 60,
                     "bundle_name": "TP_Q27_PROBE_GPU_BUNDLE.zip"},
}
if RUN_PROFILE not in PROFILE_TABLE:
    raise RuntimeError(f"Unknown RUN_PROFILE {RUN_PROFILE!r}; choose one of {list(PROFILE_TABLE)}")
PROFILE = PROFILE_TABLE[RUN_PROFILE]
USE_MODEL = bool(PROFILE["use_model"])
FAKE_ENGINE = bool(PROFILE["fake_engine"])
OUTPUT_TOKENS = int(PROFILE["output_tokens"])
PROMPT_POOL_SIZE = int(PROFILE["prompt_pool"])
PER_CONFIG_TIMEOUT_SEC = float(PROFILE["per_config_timeout_sec"])
TOTAL_BUDGET_SEC = float(PROFILE["total_budget_sec"])
BUNDLE_NAME = str(PROFILE["bundle_name"])

# ---- Kaggle inputs (identical to Q1 / PIV-Q27-ACT v1.9). TP_PROBE_INPUT_ROOT lets the CPU
# ---- selftest point at a local stand-in tree; on Kaggle leave it unset.
INPUT_ROOT = Path(os.environ.get("TP_PROBE_INPUT_ROOT", "/kaggle/input"))
MODEL_PATH = INPUT_ROOT / "models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
WHEELHOUSE = INPUT_ROOT / "datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
ARC_ROOT = INPUT_ROOT / "competitions/arc-prize-2026-arc-agi-2"
EVAL_CHALLENGES_PATH = ARC_ROOT / "arc-agi_evaluation_challenges.json"
WORKING_ROOT = Path(os.environ.get("TP_PROBE_WORKING_ROOT", "/kaggle/working"))
WORK_ROOT = WORKING_ROOT / "tp_q27_probe_v1" / RUN_PROFILE.lower()
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
for _sub in ("specs", "results", "logs", "gpu_samples", "analysis"):
    (WORK_ROOT / _sub).mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NOTEBOOK_WALL_START = time.perf_counter()
DEADLINE = NOTEBOOK_WALL_START + TOTAL_BUDGET_SEC

# ---- Q1-frozen engine (byte-for-byte the ENGINE_CONFIG of Q1 / PIV v1.7-v1.9, minus the model path).
Q1_FROZEN_ENGINE = {
    "tensor_parallel_size": 4,
    "distributed_executor_backend": "mp",
    "dtype": "auto",
    "load_format": "safetensors",
    "trust_remote_code": False,
    "language_model_only": True,
    "max_model_len": 29696,
    "max_num_seqs": 16,
    "max_num_batched_tokens": 32768,
    "gpu_memory_utilization": 0.90,
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "enable_prefix_caching": False,
    "seed": SEED,
}
MAX_MODEL_LEN = int(Q1_FROZEN_ENGINE["max_model_len"])

# ---- The ladder. Each row = frozen engine + overrides. decode_points = concurrent sequences
# ---- measured (each point is one generate() wave of OUTPUT_TOKENS tokens per sequence with
# ---- ignore_eos=True, so every sequence runs the full length and the number is a capacity
# ---- measurement, not a stop-token lottery). "optional" rows are skipped first if time is short.
_CG = {"enforce_eager": False}
_CG_AR = {"enforce_eager": False, "disable_custom_all_reduce": False}
CONFIG_LADDER = [
    {"name": "C0_frozen_q1",          "overrides": {},                                          "decode_points": [8, 16],
     "hypothesis": "Anchor. Must reproduce the Q1/PIV measurement: ~50 tok/s aggregate at 8 sequences (0.16 s/token/seq)."},
    {"name": "C1_cudagraph",          "overrides": dict(_CG),                                   "decode_points": [8, 16],
     "hypothesis": "enforce_eager=False (CUDA graphs + torch.compile). The Q1 engine was latency-bound (same step time at batch 8 and 16), so removing per-step launch overhead should give the largest single gain."},
    {"name": "C2_cudagraph_customAR", "overrides": dict(_CG_AR),                                "decode_points": [8, 16],
     "hypothesis": "Re-enable vLLM custom all-reduce. L4s are PCIe-only; expect small gain or automatic fallback to NCCL."},
    {"name": "C3_mns64",              "overrides": dict(_CG_AR, max_num_seqs=64),               "decode_points": [16, 32, 64],
     "hypothesis": "Batch scaling. If per-sequence speed stays near-flat, aggregate throughput grows ~linearly with concurrency."},
    {"name": "C4_mns128",             "overrides": dict(_CG_AR, max_num_seqs=128),              "decode_points": [64, 128],
     "hypothesis": "Upper end of batch scaling; KV-cache capacity (logged by vLLM) decides whether 128 x ~8k-token requests fit."},
    {"name": "C5_mns64_prefix",       "overrides": dict(_CG_AR, max_num_seqs=64, enable_prefix_caching=True), "decode_points": [32],
     "hypothesis": "Prefix caching on the hybrid (GDN + attention, model_type qwen3_5) checkpoint with vLLM's default mamba_cache_mode ('all' when supported): does the engine accept it, and how much faster is a warm re-prefill of the same prompt (multi-turn agent turns)?"},
    {"name": "C6_mns64_prefix_align", "overrides": dict(_CG_AR, max_num_seqs=64, enable_prefix_caching=True, mamba_cache_mode="align"), "decode_points": [32], "optional": True,
     "hypothesis": "Explicit GDN 'align' cache mode (caches SSM state only at scheduler-step boundaries; cheaper, lower hit rate). Compare warm/cold prefill and decode with C5. A rejected kwarg/value is recorded as 'unsupported', not a failure."},
    {"name": "C7_mns128_kvfp8",       "overrides": dict(_CG_AR, max_num_seqs=128, kv_cache_dtype="fp8"), "decode_points": [64, 128], "optional": True,
     "hypothesis": "FP8 KV cache doubles KV capacity (more concurrency per GPU). Numerics change; the greedy-fingerprint agreement vs C0 shows how much."},
]
if PROFILE["config_filter"]:
    CONFIG_LADDER = [c for c in CONFIG_LADDER if c["name"] in PROFILE["config_filter"]]
if PROBE_CONFIGS:
    _unknown = [n for n in PROBE_CONFIGS if n not in {c["name"] for c in CONFIG_LADDER}]
    if _unknown:
        raise RuntimeError(f"PROBE_CONFIGS names not in ladder: {_unknown}")
    CONFIG_LADDER = [c for c in CONFIG_LADDER if c["name"] in PROBE_CONFIGS]
if FAKE_ENGINE:
    # Synthetic rows that MUST fail / hang / be skipped, to prove those paths end in a bundle, not a crash.
    CONFIG_LADDER = CONFIG_LADDER + [
        {"name": "ZZ_fake_fail", "overrides": dict(_CG), "decode_points": [8], "hypothesis": "selftest: engine init raises"},
        {"name": "ZZ_fake_hang", "overrides": dict(_CG), "decode_points": [8], "hypothesis": "selftest: worker hangs -> parent timeout kill", "timeout_sec": 6},
        {"name": "ZZ_fake_skip", "overrides": dict(_CG), "decode_points": [8], "hypothesis": "selftest: projection exceeds budget -> skipped_deadline", "projected_sec_override": 1e9, "optional": True},
    ]
if not CONFIG_LADDER:
    raise RuntimeError("Empty CONFIG_LADDER after filtering.")

# ---- Measurement constants
PREFILL_PROMPTS = 32                 # prompts used for the prefill cold/warm measurement
FINGERPRINT_PROMPTS = 4              # greedy 64-token fingerprint for numerics comparison
FINGERPRINT_TOKENS = 64
WARMUP_PROMPTS = 2
GPU_SAMPLE_INTERVAL_SEC = 5.0        # nvidia-smi sampling while a config runs
PRIOR_LOAD_SEC = 420.0               # projection prior before the first measured load (Q1: 277 s + compile margin)
PRIOR_SEC_PER_OUTPUT_TOKEN = 0.16    # Q1/PIV measured per-sequence step time
CONFIG_OVERHEAD_SEC = 150.0          # warmup + prefill + fingerprint + teardown allowance
THROUGHPUT_TARGET_TPS = 300.0        # decision target for aggregate generated tok/s

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def stable_json_sha256(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def small_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str, ensure_ascii=False)

def elapsed_sec():
    return time.perf_counter() - NOTEBOOK_WALL_START

required = [MODEL_PATH, ARC_ROOT, EVAL_CHALLENGES_PATH] + ([WHEELHOUSE] if USE_MODEL else [])
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Required input is missing: {p}")
config_path = MODEL_PATH / "config.json"
index_path = MODEL_PATH / "model.safetensors.index.json"
with open(config_path, "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
qcfg = model_cfg.get("quantization_config", {})
if model_cfg.get("model_type") != "qwen3_5":
    raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
    raise RuntimeError(f"Unexpected quantization_config={qcfg}")

MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {
    "model_path": str(MODEL_PATH),
    "config_sha256": small_file_sha256(config_path),
    "safetensors_index_sha256": small_file_sha256(index_path) if index_path.exists() else None,
}
MODEL_ARTIFACT_FINGERPRINT_SHA256 = stable_json_sha256(MODEL_ARTIFACT_FINGERPRINT_PAYLOAD)

print("=" * 116)
print(EXPERIMENT_ID, "| engine-throughput ladder, one subprocess per engine config")
print("=" * 116)
print("RUN_ID:", RUN_ID, "| RUN_PROFILE:", RUN_PROFILE, "| fake_engine:", FAKE_ENGINE)
print("OUTPUT_TOKENS per sequence:", OUTPUT_TOKENS, "| prompt pool:", PROMPT_POOL_SIZE, "| budget:", TOTAL_BUDGET_SEC / 3600, "h")
print("Ladder:", [c["name"] for c in CONFIG_LADDER])
print("Q1 frozen engine sha256:", stable_json_sha256(Q1_FROZEN_ENGINE))
print("Model artifact fingerprint:", MODEL_ARTIFACT_FINGERPRINT_SHA256)
print("WORK_ROOT:", WORK_ROOT)


In [ ]:
# =====================================================================================
# CELL 2 — RUNTIME PARITY (same pinned stack as Q1 / PIV: vllm 0.19.0 from the offline wheelhouse)
# The notebook process itself never imports vllm/torch: every engine lives in a subprocess.
# =====================================================================================
EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}

def offline_pip_install(requirements):
    if not requirements:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

before_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
if USE_MODEL:
    for _pkg, _module in {"transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _module in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the Kaggle session before Run All.")
    if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        repair = [f"{p}=={v}" for p, v in EXPECTED_RUNTIME_PACKAGES.items() if p != "vllm" and before_stack.get(p) != v]
        offline_pip_install(repair)
    after_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
    for package, wanted in EXPECTED_RUNTIME_PACKAGES.items():
        if after_stack.get(package) != wanted:
            raise RuntimeError(f"Runtime parity failure: {package}={after_stack.get(package)!r}, expected {wanted!r}")
    print("Runtime dependency parity PASS:", after_stack)
else:
    after_stack = dict(before_stack)
    print("SELFTEST_CPU: no vLLM install; runtime stack as found:", after_stack)

RUNTIME_STACK = dict(after_stack)
RUNTIME_STACK["python"] = platform.python_version()
RUNTIME_STACK_SHA256 = stable_json_sha256(RUNTIME_STACK)

def nvidia_smi_query(fields, extra=()):
    """Return list of rows (list of str) or None when nvidia-smi is unavailable."""
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={','.join(fields)}", "--format=csv,noheader,nounits", *extra],
                             capture_output=True, text=True, timeout=20)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return None
    if out.returncode != 0:
        return None
    return [[c.strip() for c in line.split(",")] for line in out.stdout.strip().splitlines() if line.strip()]

GPU_INVENTORY = nvidia_smi_query(["index", "name", "memory.total", "memory.used", "driver_version"]) or []
print("GPU inventory (nvidia-smi):", GPU_INVENTORY if GPU_INVENTORY else "unavailable")
if USE_MODEL and len(GPU_INVENTORY) != 4:
    raise RuntimeError(f"Expected 4 GPUs via nvidia-smi, saw {len(GPU_INVENTORY)}. Select the Kaggle 4xL4 accelerator.")


In [ ]:
# =====================================================================================
# CELL 3 — PROMPT POOL: real ARC evaluation tasks rendered through the real Qwen chat template.
# Content is irrelevant for throughput; LENGTH is not (prefill cost, KV footprint). The pool is
# a rank-stratified sample of the public evaluation set so the prompt-token distribution matches
# what the PIV/PTQ agents actually send (Q1/PIV: min ~3k, median ~6k, max ~13k tokens).
# =====================================================================================
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
_template_path = MODEL_PATH / "chat_template.jinja"
CHAT_TEMPLATE_SHA256 = small_file_sha256(_template_path) if _template_path.exists() else None

with open(EVAL_CHALLENGES_PATH, "r", encoding="utf-8") as f:
    eval_challenges = json.load(f)

PROBE_SYSTEM_PROMPT = (
    "You are an expert ARC-AGI puzzle solver with a persistent Python tool. Inspect the training pairs, "
    "form a hypothesis about the transformation, test it in Python, and finally write a self-contained "
    "def solution(grid) function that reproduces every training output exactly."
)

def grid_to_text(grid):
    return "\n".join(" ".join(str(int(v)) for v in row) for row in grid)

def puzzle_to_text(task):
    parts = ["## Training examples"]
    for i, ex in enumerate(task["train"], 1):
        parts.append(f"### Training example {i}\nInput:\n{grid_to_text(ex['input'])}\nOutput:\n{grid_to_text(ex['output'])}")
    parts.append("## Test inputs")
    for i, ex in enumerate(task["test"], 1):
        parts.append(f"### Test input {i}\n{grid_to_text(ex['input'])}")
    return "\n\n".join(parts)

def render_prompt(task):
    messages = [{"role": "system", "content": PROBE_SYSTEM_PROMPT},
                {"role": "user", "content": puzzle_to_text(task) + "\n\nStart by inspecting the puzzle in Python."}]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        method = "chat_template"
    except Exception as e:  # template missing / incompatible: plain concatenation keeps the length realistic
        text = PROBE_SYSTEM_PROMPT + "\n\n" + messages[1]["content"] + "\n\nAssistant:"
        method = f"plain_fallback({type(e).__name__})"
    return text, method

_all_records = []
for task_id in sorted(eval_challenges):
    text, method = render_prompt(eval_challenges[task_id])
    n_tok = len(tokenizer.encode(text, add_special_tokens=False))
    _all_records.append({"task_id": task_id, "prompt": text, "prompt_tokens": int(n_tok), "render_method": method})

_fit = [r for r in _all_records if r["prompt_tokens"] + OUTPUT_TOKENS + 64 <= MAX_MODEL_LEN]
_dropped = len(_all_records) - len(_fit)
_fit.sort(key=lambda r: r["prompt_tokens"])
if len(_fit) <= PROMPT_POOL_SIZE:
    PROMPT_POOL = list(_fit)
else:
    _idx = np.linspace(0, len(_fit) - 1, PROMPT_POOL_SIZE).round().astype(int)
    PROMPT_POOL = [_fit[i] for i in sorted(set(int(i) for i in _idx))]
# Interleave short/long so any prefix of the pool (a decode point) has a representative length mix.
_rng = np.random.default_rng(SEED)
PROMPT_POOL = [PROMPT_POOL[i] for i in _rng.permutation(len(PROMPT_POOL))]

PROMPTS_PATH = WORK_ROOT / "prompts.json"
write_json(PROMPTS_PATH, PROMPT_POOL)
_lens = np.array([r["prompt_tokens"] for r in PROMPT_POOL])
PROMPT_POOL_STATS = {
    "eval_tasks_total": len(_all_records),
    "dropped_for_context": _dropped,
    "pool_size": len(PROMPT_POOL),
    "prompt_tokens_min": int(_lens.min()), "prompt_tokens_median": float(np.median(_lens)),
    "prompt_tokens_p90": float(np.percentile(_lens, 90)), "prompt_tokens_max": int(_lens.max()),
    "render_methods": sorted(set(r["render_method"] for r in PROMPT_POOL)),
    "chat_template_sha256": CHAT_TEMPLATE_SHA256,
    "prompts_sha256": small_file_sha256(PROMPTS_PATH),
}
print(json.dumps(PROMPT_POOL_STATS, indent=2))
if len(PROMPT_POOL) < 4:
    raise RuntimeError("Prompt pool too small.")


In [ ]:
# =====================================================================================
# CELL 4 — WORKER SCRIPT (one engine per subprocess) + PARENT ORCHESTRATION HELPERS
# The parent (this notebook) never imports torch/vllm, so it holds no CUDA context and every
# configuration starts from empty GPUs. Timeouts kill the whole process group, then nvidia-smi
# is polled until the GPUs are free again before the next configuration starts.
# =====================================================================================
WORKER_SRC = r'''
# tp_worker.py -- measure one vLLM engine configuration. Written by the TP-Q27 probe notebook.
import argparse, json, os, sys, time, random, traceback, dataclasses

class Unsupported(Exception):
    pass

def read_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

class FakeEngine:
    """CPU stand-in with a plausible speed model; only for SELFTEST_CPU."""
    def __init__(self, spec, result):
        kw = dict(spec["engine_kwargs"])
        name = spec["name"]
        if "fake_fail" in name:
            raise RuntimeError("selftest: simulated engine init failure (CUDA OOM during graph capture)")
        if "fake_hang" in name:
            time.sleep(3600)
        self.scale = float(spec.get("fake_time_scale", 1.0))
        base = 6.25
        if not kw.get("enforce_eager", True):
            base *= 2.6
        if not kw.get("disable_custom_all_reduce", True):
            base *= 1.08
        if kw.get("kv_cache_dtype") == "fp8":
            base *= 0.97
        self.base = base
        self.prefix = bool(kw.get("enable_prefix_caching"))
        self.seen = set()
        time.sleep(0.3 * 1.0)
        result["engine_kwargs_used"] = kw
        result["load_sec"] = 277.0 + (90.0 if not kw.get("enforce_eager", True) else 0.0)
        result["fake"] = True

    def generate(self, prompts, sp):
        n = len(prompts)
        mt = int(sp["max_tokens"])
        per_seq = self.base / (1.0 + 0.012 * n)
        prefill = 0.0
        for p in prompts:
            h = hash(p)
            rate = 40000.0 if (self.prefix and h in self.seen) else 4000.0
            prefill += (len(p) / 4.0) / rate
            self.seen.add(h)
        time.sleep((prefill + mt / per_seq) / self.scale)
        rng = random.Random(int(sp.get("seed", 0)) + n)
        outs = []
        for p in prompts:
            if float(sp.get("temperature", 1.0)) > 0:
                ids = [rng.randrange(1000, 50000) for _ in range(mt)]
            else:
                ids = [(i * 7919 + len(p)) % 50000 for i in range(mt)]
            outs.append({"prompt_token_ids": [0] * (len(p) // 4), "token_ids": ids})
        return outs

class RealEngine:
    def __init__(self, spec, result):
        import inspect
        import vllm
        from vllm import LLM, SamplingParams
        from vllm.engine.arg_utils import EngineArgs
        self.SamplingParams = SamplingParams
        result["runtime"] = {"vllm": getattr(vllm, "__version__", None)}
        valid = {f.name for f in dataclasses.fields(EngineArgs)} | set(inspect.signature(LLM.__init__).parameters)
        valid.discard("kwargs"); valid.discard("self")
        kw = dict(spec["engine_kwargs"])
        overrides = dict(spec.get("overrides", {}))
        # The Q1-frozen base kwargs are known-good on this stack; only the ladder's override keys can be
        # unsupported. A rejected override is a distinct outcome ("unsupported"), not an engine failure.
        unsupported = sorted(k for k in overrides if k not in valid)
        result["unsupported_kwargs"] = unsupported
        result["base_kwargs_not_in_signature"] = sorted(k for k in kw if k not in valid and k not in overrides)
        if unsupported:
            raise Unsupported("engine kwargs not accepted by this vLLM build: %s" % unsupported)
        result["engine_kwargs_used"] = kw
        t0 = time.perf_counter()
        try:
            self.llm = LLM(**kw)
        except Exception as e:
            msg = "%s: %s" % (type(e).__name__, e)
            if overrides and any(k in msg for k in overrides):
                result["unsupported_kwargs"] = sorted(k for k in overrides if k in msg)
                raise Unsupported(msg)
            raise
        result["load_sec"] = time.perf_counter() - t0

    def generate(self, prompts, sp):
        params = self.SamplingParams(**sp)
        outs = self.llm.generate(prompts, params, use_tqdm=False)
        return [{"prompt_token_ids": list(getattr(o, "prompt_token_ids", None) or []),
                 "token_ids": list(o.outputs[0].token_ids)} for o in outs]

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--spec", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--fake", action="store_true")
    a = ap.parse_args()
    spec = read_json(a.spec)
    scale = float(spec.get("fake_time_scale", 1.0)) if a.fake else 1.0
    result = {"config": spec["name"], "status": "started", "phases": [], "timeline": [], "error": None,
              "engine_kwargs_used": None, "unsupported_kwargs": [], "load_sec": None, "fingerprint": None,
              "worker_pid": os.getpid(), "started_epoch": time.time()}

    def dump():
        tmp = a.out + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, sort_keys=True, default=str)
        os.replace(tmp, a.out)

    def mark(event):
        result["timeline"].append({"event": event, "epoch": time.time()})
        dump()

    dump()
    pool = read_json(spec["prompts_path"])
    texts = [r["prompt"] for r in pool]
    ptoks = [int(r["prompt_tokens"]) for r in pool]
    out_tokens = int(spec["output_tokens"])
    seed = int(spec["seed"])
    engine = None

    def wave(name, idx, sp, note=None):
        ts = [texts[i % len(texts)] for i in idx]
        pt_expected = sum(ptoks[i % len(ptoks)] for i in idx)
        mark("phase_start:" + name)
        t0 = time.perf_counter()
        outs = engine.generate(ts, sp)
        wall = (time.perf_counter() - t0) * scale
        mark("phase_end:" + name)
        ot = [len(o["token_ids"]) for o in outs]
        pe = [len(o["prompt_token_ids"]) for o in outs]
        ph = {"phase": name, "n": len(ts), "wall_sec": wall, "max_tokens": int(sp["max_tokens"]),
              "out_tokens_total": sum(ot), "out_tokens_min": min(ot), "out_tokens_max": max(ot),
              "all_reached_max_tokens": all(t == int(sp["max_tokens"]) for t in ot),
              "prompt_tokens_total_tokenizer": pt_expected,
              "prompt_tokens_total_engine": (sum(pe) if all(pe) else None),
              "prompts_repeated": (max(idx) >= len(texts)) if idx else False,
              "prompt_index_range": [min(idx), max(idx)] if idx else None,
              "agg_gen_tps": sum(ot) / wall if wall > 0 else None,
              "per_seq_gen_tps": (sum(ot) / len(ot)) / wall if wall > 0 else None,
              "sec_per_output_token_per_seq": wall / max(1, int(sp["max_tokens"])),
              "prefill_tps": pt_expected / wall if wall > 0 else None,
              "note": note}
        result["phases"].append(ph)
        dump()
        return outs

    try:
        mark("load_start")
        engine = FakeEngine(spec, result) if a.fake else RealEngine(spec, result)
        mark("load_end")
        dump()
        max_seqs = int(spec["engine_kwargs"].get("max_num_seqs", 16))
        base_sp = {"temperature": 1.0, "top_p": 0.95, "top_k": 20, "seed": seed, "ignore_eos": True}
        pool_len = len(texts)
        # warmup (also validates generate() on this engine)
        n_warm = min(int(spec["warmup_prompts"]), pool_len)
        wave("warmup", list(range(n_warm)), dict(base_sp, max_tokens=16))
        # decode capacity points. Each point takes the NEXT slice of the pool so that, with prefix
        # caching on, a later point does not re-prefill prompts an earlier point already cached.
        # (Slices wrap and are flagged 'prompts_repeated' when the pool is smaller than the demand.)
        start = n_warm
        for n in sorted(int(x) for x in spec["decode_points"]):
            if n > max_seqs:
                result["phases"].append({"phase": "decode@%d" % n, "n": n, "skipped": "n > max_num_seqs=%d" % max_seqs})
                dump()
                continue
            idx = [start + j for j in range(n)]
            wave("decode@%d" % n, idx, dict(base_sp, max_tokens=out_tokens),
                 note=("pool slice wraps (prompts reused across points)" if start + n > pool_len else None))
            start += n
        # prefill cold vs warm: prompts NOT used by any decode point when the pool allows it, otherwise
        # flagged (a prefix-caching engine would already hold them, making 'cold' warm).
        k = min(int(spec["prefill_prompts"]), pool_len, max_seqs)
        fresh = pool_len - min(start, pool_len)
        if fresh >= max(4, k // 2):
            k = min(k, fresh)
            pre_idx = list(range(pool_len - k, pool_len))
            overlap_note = None
        else:
            pre_idx = list(range(k))
            overlap_note = "prefill prompts overlap decode prompts (pool too small); cold may already be cached"
        pre_sp = {"temperature": 0.0, "max_tokens": 1, "ignore_eos": True}
        wave("prefill_cold", pre_idx, pre_sp, note=overlap_note)
        wave("prefill_warm", pre_idx, pre_sp, note="identical prompts re-sent" + ("; " + overlap_note if overlap_note else ""))
        # greedy fingerprint for cross-config numerics comparison
        fk = min(int(spec["fingerprint_prompts"]), len(texts))
        fp_sp = {"temperature": 0.0, "max_tokens": int(spec["fingerprint_tokens"]), "ignore_eos": True, "seed": seed}
        outs = wave("fingerprint", list(range(fk)), fp_sp)
        result["fingerprint"] = [o["token_ids"] for o in outs]
        result["status"] = "ok"
    except Unsupported as e:
        result["status"] = "unsupported"
        result["error"] = str(e)
    except BaseException as e:
        result["status"] = "failed"
        result["error"] = "%s: %s" % (type(e).__name__, e)
        result["traceback"] = traceback.format_exc()
    result["finished_epoch"] = time.time()
    dump()
    code = 0 if result["status"] in ("ok", "unsupported") else 3
    print("TP_WORKER_DONE status=%s config=%s" % (result["status"], spec["name"]), flush=True)
    # Prefer a clean exit (vLLM's atexit hooks shut the TP workers down); if that hangs, hard-exit
    # after 90 s. The parent reclaims any GPU process that survives either path.
    import threading
    threading.Thread(target=lambda: (time.sleep(90.0), os._exit(code)), daemon=True).start()
    engine = None
    import gc
    gc.collect()
    sys.stdout.flush(); sys.stderr.flush()
    sys.exit(code)

if __name__ == "__main__":
    main()
'''
WORKER_PATH = WORK_ROOT / "tp_worker.py"
WORKER_PATH.write_text(WORKER_SRC, encoding="utf-8")
WORKER_SHA256 = small_file_sha256(WORKER_PATH)
import py_compile
py_compile.compile(str(WORKER_PATH), doraise=True)
print("worker script:", WORKER_PATH, "sha256", WORKER_SHA256[:16], "compile OK")

# ---------------------------------------------------------------- parent helpers
def gpu_sample():
    rows = nvidia_smi_query(["index", "memory.used", "utilization.gpu"])
    return {"epoch": time.time(), "gpus": rows}

def gpu_compute_pids():
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=20)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return None
    if out.returncode != 0:
        return None
    pids = []
    for line in out.stdout.strip().splitlines():
        parts = [c.strip() for c in line.split(",")]
        if parts and parts[0].isdigit():
            pids.append((int(parts[0]), parts[1] if len(parts) > 1 else None))
    return pids

def gpus_are_free(threshold_mib=1500):
    rows = nvidia_smi_query(["index", "memory.used"])
    if rows is None:
        return True
    try:
        return all(float(r[1]) < threshold_mib for r in rows)
    except (ValueError, IndexError):
        return True

def kill_process_tree(proc):
    """SIGKILL the worker's whole process group (posix) or the worker (Windows)."""
    try:
        if os.name == "nt":
            proc.kill()
        else:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
    except (ProcessLookupError, PermissionError, OSError):
        try:
            proc.kill()
        except Exception:
            pass

def reclaim_gpus(max_wait_sec=120.0):
    """After a config exits: kill any leftover CUDA process (the notebook itself owns none), wait until free."""
    info = {"leftover_pids_killed": [], "free_after_sec": None, "free": None}
    t0 = time.perf_counter()
    while True:
        pids = gpu_compute_pids()
        if pids:
            for pid, mem in pids:
                if pid != os.getpid():
                    try:
                        os.kill(pid, signal.SIGKILL if os.name != "nt" else signal.SIGTERM)
                        info["leftover_pids_killed"].append({"pid": pid, "used_mib": mem})
                    except (ProcessLookupError, PermissionError, OSError):
                        pass
        if gpus_are_free():
            info["free"] = True
            info["free_after_sec"] = time.perf_counter() - t0
            return info
        if time.perf_counter() - t0 > max_wait_sec:
            info["free"] = False
            info["free_after_sec"] = time.perf_counter() - t0
            return info
        time.sleep(3.0)

LOG_PATTERNS = {
    "kv_cache_tokens":        re.compile(r"GPU KV cache size:\s*([\d,]+)\s*tokens"),
    "max_concurrency_x":      re.compile(r"Maximum concurrency for\s*([\d,]+)\s*tokens per request:\s*([\d.]+)x"),
    "graph_capture_sec":      re.compile(r"Graph capturing finished in\s*([\d.]+)\s*secs"),
    "torch_compile_total_sec": re.compile(r"torch\.compile takes\s*([\d.]+)\s*s in total"),
    "model_loading":          re.compile(r"Model loading took\s*([\d.]+)\s*GiB memory and\s*([\d.]+)\s*seconds"),
}
ERROR_HINT = re.compile(r"(CUDA out of memory|OutOfMemory|torch\.OutOfMemoryError|RuntimeError|ValueError|NotImplementedError|Error:)")
CACHE_HINT = re.compile(r"(prefix caching|prefix cache|mamba|Mamba|cudagraph|CUDA graph|custom allreduce|custom all-reduce|Custom allreduce|kv_cache_dtype|KV cache dtype)", re.IGNORECASE)

def parse_worker_log(log_path):
    facts = {"kv_cache_tokens": None, "max_concurrency_x": None, "max_concurrency_len": None, "graph_capture_sec": None,
             "torch_compile_total_sec": None, "model_loading_gib": None, "model_loading_sec": None, "error_hints": [],
             "engine_feature_lines": [], "log_lines": 0}
    if not Path(log_path).exists():
        return facts
    with open(log_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            facts["log_lines"] += 1
            m = LOG_PATTERNS["kv_cache_tokens"].search(line)
            if m: facts["kv_cache_tokens"] = int(m.group(1).replace(",", ""))
            m = LOG_PATTERNS["max_concurrency_x"].search(line)
            if m: facts["max_concurrency_len"], facts["max_concurrency_x"] = int(m.group(1).replace(",", "")), float(m.group(2))
            m = LOG_PATTERNS["graph_capture_sec"].search(line)
            if m: facts["graph_capture_sec"] = float(m.group(1))
            m = LOG_PATTERNS["torch_compile_total_sec"].search(line)
            if m: facts["torch_compile_total_sec"] = float(m.group(1))
            m = LOG_PATTERNS["model_loading"].search(line)
            if m: facts["model_loading_gib"], facts["model_loading_sec"] = float(m.group(1)), float(m.group(2))
            if ERROR_HINT.search(line) and len(facts["error_hints"]) < 12:
                facts["error_hints"].append(line.rstrip()[:300])
            if CACHE_HINT.search(line) and len(facts["engine_feature_lines"]) < 40:
                facts["engine_feature_lines"].append(line.rstrip()[:300])
    return facts

def build_spec(cfg):
    engine_kwargs = dict(Q1_FROZEN_ENGINE)
    engine_kwargs["model"] = str(MODEL_PATH)
    engine_kwargs.update(cfg["overrides"])
    return {
        "name": cfg["name"], "hypothesis": cfg.get("hypothesis"), "optional": bool(cfg.get("optional", False)),
        "engine_kwargs": engine_kwargs, "overrides": dict(cfg["overrides"]),
        "decode_points": list(cfg["decode_points"]), "output_tokens": OUTPUT_TOKENS, "seed": SEED,
        "prompts_path": str(PROMPTS_PATH), "warmup_prompts": WARMUP_PROMPTS, "prefill_prompts": PREFILL_PROMPTS,
        "fingerprint_prompts": FINGERPRINT_PROMPTS, "fingerprint_tokens": FINGERPRINT_TOKENS,
        "fake_time_scale": 200.0 if FAKE_ENGINE else 1.0,
        "timeout_sec": float(cfg.get("timeout_sec", PER_CONFIG_TIMEOUT_SEC)),
    }

def run_config(cfg):
    spec = build_spec(cfg)
    name = cfg["name"]
    spec_path = WORK_ROOT / "specs" / f"{name}.json"
    out_path = WORK_ROOT / "results" / f"{name}.json"
    log_path = WORK_ROOT / "logs" / f"{name}.log"
    samples_path = WORK_ROOT / "gpu_samples" / f"{name}.jsonl"
    write_json(spec_path, spec)
    env = dict(os.environ)
    env["VLLM_LOGGING_LEVEL"] = "INFO"
    cmd = [sys.executable, str(WORKER_PATH), "--spec", str(spec_path), "--out", str(out_path)] + (["--fake"] if FAKE_ENGINE else [])
    popen_kwargs = {}
    if os.name == "nt":
        popen_kwargs["creationflags"] = getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0)
    else:
        popen_kwargs["start_new_session"] = True
    record = {"config": name, "spec": spec, "launched_epoch": time.time(), "status": None, "timed_out": False,
              "returncode": None, "parent_wall_sec": None, "result": None, "log_facts": None, "gpu_reclaim": None}
    t0 = time.perf_counter()
    last_sample = 0.0
    with open(log_path, "w", encoding="utf-8") as logf, open(samples_path, "w", encoding="utf-8") as sf:
        proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env, cwd=str(WORK_ROOT), **popen_kwargs)
        while True:
            rc = proc.poll()
            now = time.perf_counter()
            if now - last_sample >= GPU_SAMPLE_INTERVAL_SEC:
                s = gpu_sample()
                if s["gpus"] is not None:
                    sf.write(json.dumps(s) + "\n"); sf.flush()
                last_sample = now
            if rc is not None:
                break
            if now - t0 > spec["timeout_sec"]:
                kill_process_tree(proc)
                try:
                    proc.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    pass
                record["timed_out"] = True
                rc = proc.returncode
                break
            time.sleep(1.0)
    record["returncode"] = rc
    record["parent_wall_sec"] = time.perf_counter() - t0
    record["gpu_reclaim"] = reclaim_gpus()
    if out_path.exists():
        with open(out_path, "r", encoding="utf-8") as f:
            record["result"] = json.load(f)
    record["log_facts"] = parse_worker_log(log_path)
    res_status = (record["result"] or {}).get("status")
    if record["timed_out"]:
        record["status"] = "timeout"
    elif res_status in ("ok", "unsupported", "failed"):
        record["status"] = res_status
    elif record["result"] is None:
        record["status"] = "no_result"
    else:
        record["status"] = "died_mid_run"   # worker started/partial phases but never wrote a terminal status
    write_json(WORK_ROOT / "results" / f"{name}.record.json", record)
    return record

def measured_speed_prior(records):
    """(est_load_sec, est_sec_per_output_token) from the latest successful config, else priors."""
    load, spt = PRIOR_LOAD_SEC, PRIOR_SEC_PER_OUTPUT_TOKEN
    for rec in records:
        r = rec.get("result") or {}
        if rec.get("status") == "ok":
            if r.get("load_sec"):
                load = float(r["load_sec"]) + 30.0
            dec = [p for p in r.get("phases", []) if str(p.get("phase", "")).startswith("decode@") and p.get("sec_per_output_token_per_seq")]
            if dec:
                spt = min(float(p["sec_per_output_token_per_seq"]) for p in dec)
    return load, spt

def project_config_sec(cfg, est_load, est_spt):
    if cfg.get("projected_sec_override") is not None:      # selftest hook
        return float(cfg["projected_sec_override"])
    pts = [int(n) for n in cfg["decode_points"] if int(n) <= int(cfg["overrides"].get("max_num_seqs", Q1_FROZEN_ENGINE["max_num_seqs"]))]
    decode = sum(OUTPUT_TOKENS * est_spt * (1.0 + 0.01 * n) for n in pts)
    return est_load + decode + CONFIG_OVERHEAD_SEC

print("parent helpers ready; GPUs free now:", gpus_are_free(), "| compute pids:", gpu_compute_pids())


In [ ]:
# =====================================================================================
# CELL 5 — RUN THE LADDER (one subprocess per config; deadline-projected; failures recorded)
# =====================================================================================
RECORDS = []
LADDER_LOG = []

def _fmt_tps(x):
    return "-" if x is None else f"{x:7.1f}"

def print_config_row(rec):
    r = rec.get("result") or {}
    dec = {p["phase"]: p for p in r.get("phases", []) if str(p.get("phase", "")).startswith("decode@") and not p.get("skipped")}
    parts = [f"{rec['config']:<24} {rec['status']:<12} load={_fmt_tps(r.get('load_sec'))}s"]
    for ph, p in sorted(dec.items(), key=lambda kv: kv[1]["n"]):
        parts.append(f"{ph}: agg={_fmt_tps(p['agg_gen_tps'])} tok/s per_seq={p['per_seq_gen_tps']:.2f}")
    lf = rec.get("log_facts") or {}
    if lf.get("kv_cache_tokens"):
        parts.append(f"KV={lf['kv_cache_tokens']:,} tok ({lf.get('max_concurrency_x')}x @ {lf.get('max_concurrency_len')})")
    if rec["status"] not in ("ok",):
        err = (r.get("error") or "")[:160]
        hints = (lf.get("error_hints") or [])[-2:]
        parts.append(f"error={err!r} hints={hints}")
    print("  " + " | ".join(parts), flush=True)

_required_remaining = [c for c in CONFIG_LADDER if not c.get("optional")]
for i, cfg in enumerate(CONFIG_LADDER):
    est_load, est_spt = measured_speed_prior(RECORDS)
    proj = project_config_sec(cfg, est_load, est_spt)
    if not cfg.get("optional"):
        _required_remaining = [c for c in _required_remaining if c["name"] != cfg["name"]]
    reserve_for_required = sum(project_config_sec(c, est_load, est_spt) for c in _required_remaining) if cfg.get("optional") else 0.0
    now = time.perf_counter()
    remaining = DEADLINE - now
    entry = {"config": cfg["name"], "index": i, "projected_sec": proj, "remaining_sec": remaining,
             "reserve_for_required_sec": reserve_for_required, "optional": bool(cfg.get("optional"))}
    if proj + reserve_for_required > remaining:
        entry["decision"] = "skipped_deadline"
        LADDER_LOG.append(entry)
        rec = {"config": cfg["name"], "spec": build_spec(cfg), "status": "skipped_deadline", "timed_out": False,
               "returncode": None, "parent_wall_sec": 0.0, "result": None, "log_facts": None, "gpu_reclaim": None,
               "skip_reason": f"projected {proj/60:.1f} min (+{reserve_for_required/60:.1f} min reserved for required configs) > remaining {remaining/60:.1f} min"}
        write_json(WORK_ROOT / "results" / f"{cfg['name']}.record.json", rec)
        RECORDS.append(rec)
        print(f"[{i+1}/{len(CONFIG_LADDER)}] {cfg['name']}: SKIPPED ({rec['skip_reason']})", flush=True)
        continue
    entry["decision"] = "run"
    LADDER_LOG.append(entry)
    print(f"[{i+1}/{len(CONFIG_LADDER)}] {cfg['name']}  projected {proj/60:.1f} min | remaining {remaining/60:.1f} min | overrides={cfg['overrides']}", flush=True)
    print(f"    hypothesis: {cfg.get('hypothesis')}", flush=True)
    rec = run_config(cfg)
    RECORDS.append(rec)
    print_config_row(rec)
    if rec["gpu_reclaim"] and rec["gpu_reclaim"].get("free") is False:
        print("    WARNING: GPUs still not free after reclaim; later configs may OOM.", flush=True)

write_json(WORK_ROOT / "analysis" / "ladder_log.json", LADDER_LOG)
print("\nLadder finished in", f"{elapsed_sec()/60:.1f} min;", {r["status"] for r in RECORDS})


In [ ]:
# =====================================================================================
# CELL 6 — ANALYSIS: summary table, anchor check, numerics agreement, verdict, projections, REPORT.md
# =====================================================================================
import pandas as pd

def _decode_phases(rec):
    r = rec.get("result") or {}
    return sorted([p for p in r.get("phases", []) if str(p.get("phase", "")).startswith("decode@") and not p.get("skipped")], key=lambda p: p["n"])

def _phase(rec, name):
    r = rec.get("result") or {}
    for p in r.get("phases", []):
        if p.get("phase") == name and not p.get("skipped"):
            return p
    return None

def _peak_gpu_mib(name):
    path = WORK_ROOT / "gpu_samples" / f"{name}.jsonl"
    peak = {}
    if not path.exists():
        return peak
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                s = json.loads(line)
            except json.JSONDecodeError:
                continue
            for row in s.get("gpus") or []:
                try:
                    idx, used = int(row[0]), float(row[1])
                except (ValueError, IndexError):
                    continue
                peak[idx] = max(peak.get(idx, 0.0), used)
    return peak

def _fingerprint_agreement(a, b):
    """Mean over prompts of (identical leading tokens / length) and exact-equality flag."""
    if not a or not b or len(a) != len(b):
        return None
    prefix_frac, exact = [], []
    for x, y in zip(a, b):
        n = min(len(x), len(y))
        k = 0
        while k < n and x[k] == y[k]:
            k += 1
        prefix_frac.append(k / n if n else 0.0)
        exact.append(x == y)
    return {"mean_prefix_agreement": float(sum(prefix_frac) / len(prefix_frac)), "prompts_identical": int(sum(exact)), "prompts": len(exact)}

ANCHOR_NAME = "C0_frozen_q1"
_anchor = next((r for r in RECORDS if r["config"] == ANCHOR_NAME and r["status"] == "ok"), None)
_anchor_fp = ((_anchor or {}).get("result") or {}).get("fingerprint")

rows = []
for rec in RECORDS:
    r = rec.get("result") or {}
    lf = rec.get("log_facts") or {}
    dec = _decode_phases(rec)
    best_pt = max(dec, key=lambda p: p["agg_gen_tps"] or 0.0) if dec else None
    cold, warm = _phase(rec, "prefill_cold"), _phase(rec, "prefill_warm")
    peak = _peak_gpu_mib(rec["config"])
    row = {
        "config": rec["config"], "status": rec["status"], "optional": bool(rec["spec"].get("optional")),
        "overrides": json.dumps(rec["spec"]["overrides"], sort_keys=True),
        "load_sec": r.get("load_sec"), "graph_capture_sec": lf.get("graph_capture_sec"), "torch_compile_sec": lf.get("torch_compile_total_sec"),
        "kv_cache_tokens": lf.get("kv_cache_tokens"), "max_concurrency_x": lf.get("max_concurrency_x"),
        "best_point_n": best_pt["n"] if best_pt else None,
        "best_agg_gen_tps": best_pt["agg_gen_tps"] if best_pt else None,
        "best_per_seq_gen_tps": best_pt["per_seq_gen_tps"] if best_pt else None,
        "best_point_prompts_repeated": bool(best_pt["prompts_repeated"]) if best_pt else None,
        "all_reached_max_tokens": all(p["all_reached_max_tokens"] for p in dec) if dec else None,
        "prefill_cold_tps": cold["prefill_tps"] if cold else None,
        "prefill_warm_tps": warm["prefill_tps"] if warm else None,
        "prefill_warm_over_cold": (warm["prefill_tps"] / cold["prefill_tps"]) if (cold and warm and cold["prefill_tps"]) else None,
        "peak_gpu_mib_max": max(peak.values()) if peak else None,
        "fingerprint_vs_anchor": _fingerprint_agreement(_anchor_fp, r.get("fingerprint")) if (_anchor_fp and r.get("fingerprint")) else None,
        "parent_wall_sec": rec.get("parent_wall_sec"), "error": (r.get("error") or rec.get("skip_reason")),
        "unsupported_kwargs": r.get("unsupported_kwargs"),
    }
    for p in dec:
        row[f"agg_tps@{p['n']}"] = p["agg_gen_tps"]
        row[f"per_seq_tps@{p['n']}"] = p["per_seq_gen_tps"]
        # decode-only estimate: subtract the prefill time implied by this config's own cold prefill rate
        p["decode_only_tps_est"] = None
        if cold and cold.get("prefill_tps") and p.get("wall_sec"):
            _pre = p["prompt_tokens_total_tokenizer"] / cold["prefill_tps"]
            if 0 < _pre < 0.8 * p["wall_sec"]:       # otherwise the prefill estimate is not trustworthy
                p["decode_only_tps_est"] = p["out_tokens_total"] / (p["wall_sec"] - _pre)
        row[f"decode_only_tps@{p['n']}"] = p["decode_only_tps_est"]
    rows.append(row)

SUMMARY = pd.DataFrame(rows)
SUMMARY.to_csv(WORK_ROOT / "analysis" / "summary_table.csv", index=False)
_show_cols = ["config", "status", "load_sec", "kv_cache_tokens", "max_concurrency_x", "best_point_n", "best_agg_gen_tps", "best_per_seq_gen_tps", "prefill_cold_tps", "prefill_warm_over_cold", "peak_gpu_mib_max"]
print(SUMMARY[_show_cols].to_string(index=False))

# ---- anchor check (was the frozen engine reproduced?)
ANCHOR = {"expected_agg_tps_at_8": 51.0, "expected_sec_per_token": PRIOR_SEC_PER_OUTPUT_TOKEN, "measured_agg_tps_at_8": None, "ratio": None, "pass": None}
if _anchor:
    p8 = next((p for p in _decode_phases(_anchor) if p["n"] == 8), None)
    if p8:
        ANCHOR["measured_agg_tps_at_8"] = p8["agg_gen_tps"]
        ANCHOR["ratio"] = p8["agg_gen_tps"] / ANCHOR["expected_agg_tps_at_8"]
        ANCHOR["pass"] = 0.6 <= ANCHOR["ratio"] <= 1.6
print("\nAnchor (C0 @8 vs Q1/PIV ~51 tok/s):", json.dumps(ANCHOR))

# ---- verdict
ok_rows = [r for r in rows if r["status"] == "ok" and r["best_agg_gen_tps"]]
valid_rows = [r for r in ok_rows if r["all_reached_max_tokens"]]
best = max(valid_rows, key=lambda r: r["best_agg_gen_tps"]) if valid_rows else None
anchor_tps = (_anchor and next((r["best_agg_gen_tps"] for r in rows if r["config"] == ANCHOR_NAME), None)) or None
best_rec = next((rec for rec in RECORDS if best and rec["config"] == best["config"]), None)
VERDICT = {
    "run_id": RUN_ID, "run_profile": RUN_PROFILE, "fake_engine": FAKE_ENGINE, "output_tokens": OUTPUT_TOKENS,
    "target_agg_tps": THROUGHPUT_TARGET_TPS,
    "anchor": ANCHOR,
    "best_config": best["config"] if best else None,
    "best_agg_gen_tps": best["best_agg_gen_tps"] if best else None,
    "best_point_n": best["best_point_n"] if best else None,
    "best_per_seq_gen_tps": best["best_per_seq_gen_tps"] if best else None,
    "speedup_vs_frozen_q1": (best["best_agg_gen_tps"] / anchor_tps) if (best and anchor_tps) else None,
    "target_met": (best["best_agg_gen_tps"] >= THROUGHPUT_TARGET_TPS) if best else False,
    "recommended_engine_kwargs": ((best_rec or {}).get("result") or {}).get("engine_kwargs_used") if best_rec else None,
    "recommended_overrides": json.loads(best["overrides"]) if best else None,
    "numerics_vs_anchor": best["fingerprint_vs_anchor"] if best else None,
    "prefix_caching": {r["config"]: {"status": r["status"], "warm_over_cold": r["prefill_warm_over_cold"], "unsupported": r["unsupported_kwargs"]} for r in rows if "prefix" in r["config"]},
    "statuses": {r["config"]: r["status"] for r in rows},
    "ladder_elapsed_min": elapsed_sec() / 60.0,
}
# ---- projections at the best measured aggregate throughput
def _hours(tokens, tps):
    return tokens / tps / 3600.0 if tps else None
PROJECTION = []
if best:
    tps = best["best_agg_gen_tps"]
    for tasks, label in ((120, "public_eval_120"), (240, "hidden_test_240")):
        for K in (1, 2, 4, 8, 16):
            for L in (8192, 12288):
                PROJECTION.append({"set": label, "tasks": tasks, "samples_per_task": K, "gen_tokens_per_sample": L,
                                   "total_gen_tokens": tasks * K * L, "hours_at_best_tps": _hours(tasks * K * L, tps),
                                   "hours_at_frozen_q1": _hours(tasks * K * L, anchor_tps) if anchor_tps else None})
VERDICT["projection_note"] = "generation-only hours = tasks x K x L / best aggregate tok/s; add ~7 min model load; prefill and tool time not included"
write_json(WORK_ROOT / "analysis" / "throughput_verdict.json", VERDICT)
pd.DataFrame(PROJECTION).to_csv(WORK_ROOT / "analysis" / "projection.csv", index=False)
print("\nVERDICT:", json.dumps({k: VERDICT[k] for k in ("best_config", "best_agg_gen_tps", "best_point_n", "speedup_vs_frozen_q1", "target_met")}))

# ---- REPORT.md
def _f(x, fmt="{:.1f}"):
    return "-" if x is None else fmt.format(x)
lines = [f"# {EXPERIMENT_ID} — {RUN_PROFILE} — run {RUN_ID}", "",
         f"Model: `{MODEL_PATH}` (fingerprint {MODEL_ARTIFACT_FINGERPRINT_SHA256[:16]}) | runtime {RUNTIME_STACK} | fake_engine={FAKE_ENGINE}", "",
         f"Output tokens per sequence: {OUTPUT_TOKENS} (ignore_eos) | prompt pool {PROMPT_POOL_STATS['pool_size']} tasks, prompt tokens min/median/p90/max = "
         f"{PROMPT_POOL_STATS['prompt_tokens_min']}/{PROMPT_POOL_STATS['prompt_tokens_median']:.0f}/{PROMPT_POOL_STATS['prompt_tokens_p90']:.0f}/{PROMPT_POOL_STATS['prompt_tokens_max']}", "",
         "## Ladder", "", "| config | status | load s | KV tokens | max conc | best n | agg tok/s | per-seq tok/s | prefill cold tok/s | warm/cold | peak GPU MiB | numerics vs C0 | error |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
for r in rows:
    fp = r["fingerprint_vs_anchor"]
    fps = "-" if not fp else f"{fp['mean_prefix_agreement']:.2f} ({fp['prompts_identical']}/{fp['prompts']} identical)"
    lines.append(f"| {r['config']} | {r['status']} | {_f(r['load_sec'])} | {_f(r['kv_cache_tokens'], '{:,}')} | {_f(r['max_concurrency_x'], '{:.1f}')} | {_f(r['best_point_n'], '{}')} | "
                 f"{_f(r['best_agg_gen_tps'])} | {_f(r['best_per_seq_gen_tps'], '{:.2f}')} | {_f(r['prefill_cold_tps'])} | {_f(r['prefill_warm_over_cold'], '{:.2f}')} | "
                 f"{_f(r['peak_gpu_mib_max'], '{:.0f}')} | {fps} | {(r['error'] or '')[:80]} |")
lines += ["", "## Per-point detail", ""]
for rec in RECORDS:
    dec = _decode_phases(rec)
    if dec:
        lines.append(f"- **{rec['config']}**: " + "; ".join(f"n={p['n']}: agg {p['agg_gen_tps']:.1f} tok/s (decode-only est {_f(p.get('decode_only_tps_est'))}), per-seq {p['per_seq_gen_tps']:.2f} tok/s, wall {p['wall_sec']:.0f}s"
                                                       + (" (prompts repeated)" if p['prompts_repeated'] else "") + ("" if p['all_reached_max_tokens'] else " (NOT all sequences reached max_tokens)") for p in dec))
lines += ["", "## Anchor check", "", f"`{json.dumps(ANCHOR)}`", "", "## Verdict", "",
          f"- best config: **{VERDICT['best_config']}** at n={VERDICT['best_point_n']}: **{_f(VERDICT['best_agg_gen_tps'])} tok/s aggregate**, {_f(VERDICT['best_per_seq_gen_tps'], '{:.2f}')} tok/s per sequence",
          f"- speed-up vs frozen Q1 engine: {_f(VERDICT['speedup_vs_frozen_q1'], '{:.2f}')}x | target {THROUGHPUT_TARGET_TPS:.0f} tok/s met: **{VERDICT['target_met']}**",
          f"- recommended overrides: `{json.dumps(VERDICT['recommended_overrides'])}`",
          f"- prefix caching: `{json.dumps(VERDICT['prefix_caching'])}`", "",
          "## Projection (generation-only hours at best aggregate tok/s)", "", "| set | K samples/task | tokens/sample | total tokens | hours @best | hours @frozen Q1 |", "|---|---|---|---|---|---|"]
for p in PROJECTION:
    lines.append(f"| {p['set']} | {p['samples_per_task']} | {p['gen_tokens_per_sample']} | {p['total_gen_tokens']:,} | {_f(p['hours_at_best_tps'], '{:.2f}')} | {_f(p['hours_at_frozen_q1'], '{:.2f}')} |")
lines += ["", "## What this run does NOT measure", "",
          "- Solve quality: outputs are ignore_eos token streams, never parsed. Engine flags do not change the model's policy; the greedy fingerprint column shows how far kernel numerics drift.",
          "- Multi-turn agent wall-clock (tool execution, verifier) — only generation and prefill.",
          "- Anything about which decode point is *needed*: that comes from the sampling recipe (K samples/task), see projection table."]
REPORT_PATH = WORK_ROOT / "REPORT.md"
REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print("\nREPORT:", REPORT_PATH)


In [ ]:
# =====================================================================================
# CELL 7 — BUNDLE: run_metadata + MANIFEST (sha256) + SELFCHECK, zipped to /kaggle/working
# =====================================================================================
import zipfile

RUN_METADATA = {
    "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID, "run_profile": RUN_PROFILE, "profile": PROFILE,
    "fake_engine": FAKE_ENGINE, "seed": SEED, "output_tokens": OUTPUT_TOKENS,
    "q1_frozen_engine": Q1_FROZEN_ENGINE, "q1_frozen_engine_sha256": stable_json_sha256(Q1_FROZEN_ENGINE),
    "config_ladder": CONFIG_LADDER, "probe_configs_override": PROBE_CONFIGS,
    "model_artifact_fingerprint": MODEL_ARTIFACT_FINGERPRINT_PAYLOAD, "model_artifact_fingerprint_sha256": MODEL_ARTIFACT_FINGERPRINT_SHA256,
    "runtime_stack": RUNTIME_STACK, "runtime_stack_sha256": RUNTIME_STACK_SHA256,
    "gpu_inventory": GPU_INVENTORY, "prompt_pool_stats": PROMPT_POOL_STATS, "worker_sha256": WORKER_SHA256,
    "measurement_constants": {"PREFILL_PROMPTS": PREFILL_PROMPTS, "FINGERPRINT_PROMPTS": FINGERPRINT_PROMPTS, "FINGERPRINT_TOKENS": FINGERPRINT_TOKENS,
                              "WARMUP_PROMPTS": WARMUP_PROMPTS, "GPU_SAMPLE_INTERVAL_SEC": GPU_SAMPLE_INTERVAL_SEC, "THROUGHPUT_TARGET_TPS": THROUGHPUT_TARGET_TPS,
                              "PER_CONFIG_TIMEOUT_SEC": PER_CONFIG_TIMEOUT_SEC, "TOTAL_BUDGET_SEC": TOTAL_BUDGET_SEC},
    "statuses": {r["config"]: r["status"] for r in RECORDS},
    "elapsed_sec": elapsed_sec(), "platform": platform.platform(), "python": platform.python_version(),
}
write_json(WORK_ROOT / "run_metadata.json", RUN_METADATA)

# Manifest over everything except the (large) prompt texts' duplicates inside specs -> keep all; prompts.json once.
files = sorted(p for p in WORK_ROOT.rglob("*") if p.is_file() and p.name != "MANIFEST.json")
MANIFEST = {"run_id": RUN_ID, "files": [{"path": str(p.relative_to(WORK_ROOT)).replace("\\", "/"), "bytes": p.stat().st_size, "sha256": small_file_sha256(p)} for p in files]}
write_json(WORK_ROOT / "MANIFEST.json", MANIFEST)

expected_results = {c["name"] for c in CONFIG_LADDER}
have_records = {p.name[:-len(".record.json")] for p in (WORK_ROOT / "results").glob("*.record.json")}
SELFCHECK = {
    "all_configs_have_record": expected_results <= have_records,
    "missing_records": sorted(expected_results - have_records),
    "statuses": RUN_METADATA["statuses"],
    "anchor_pass": ANCHOR.get("pass"),
    "best_config": VERDICT.get("best_config"),
    "target_met": VERDICT.get("target_met"),
    "report_present": REPORT_PATH.exists(),
    "manifest_files": len(MANIFEST["files"]),
}
if FAKE_ENGINE:
    SELFCHECK["selftest_expectations"] = {
        "fake_fail_recorded_as_failed": RUN_METADATA["statuses"].get("ZZ_fake_fail") == "failed",
        "fake_hang_recorded_as_timeout": RUN_METADATA["statuses"].get("ZZ_fake_hang") == "timeout",
        "fake_skip_recorded_as_skipped": RUN_METADATA["statuses"].get("ZZ_fake_skip") == "skipped_deadline",
        "ladder_configs_ok": all(v == "ok" for k, v in RUN_METADATA["statuses"].items() if not k.startswith("ZZ_")),
        "anchor_and_verdict_present": bool(VERDICT.get("best_config")) and ANCHOR.get("measured_agg_tps_at_8") is not None,
        "every_ok_config_has_all_phases": all(
            {"warmup", "prefill_cold", "prefill_warm", "fingerprint"} <= {p.get("phase") for p in (r.get("result") or {}).get("phases", [])}
            and any(str(p.get("phase", "")).startswith("decode@") and not p.get("skipped") for p in (r.get("result") or {}).get("phases", []))
            for r in RECORDS if r["status"] == "ok"),
        "timeout_left_no_gpu_process": all((r.get("gpu_reclaim") or {}).get("free") in (True, None) for r in RECORDS),
    }
    SELFCHECK["selftest_pass"] = all(SELFCHECK["selftest_expectations"].values()) and SELFCHECK["all_configs_have_record"]
write_json(WORK_ROOT / "SELFCHECK.json", SELFCHECK)

BUNDLE_PATH = WORKING_ROOT / BUNDLE_NAME
if BUNDLE_PATH.exists():
    BUNDLE_PATH.unlink()
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORK_ROOT.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(WORK_ROOT)).replace("\\", "/"))
print("Bundle:", BUNDLE_PATH, f"({BUNDLE_PATH.stat().st_size/1024:.0f} KB, {len(MANIFEST['files'])} files)")
print("SELFCHECK:", json.dumps(SELFCHECK, indent=2))
if FAKE_ENGINE and not SELFCHECK["selftest_pass"]:
    raise RuntimeError("SELFTEST_CPU failed; see SELFCHECK above.")


## Run sequence

1. **SELFTEST_CPU** — set `RUN_PROFILE = "SELFTEST_CPU"`, run all cells (PC with `TP_PROBE_INPUT_ROOT` pointing at a
   stand-in tree, or a Kaggle CPU session with the model dataset + competition data attached). Must end with
   `SELFCHECK.selftest_pass = true`: the two `ZZ_*` rows prove the failure and timeout paths.
2. **SMOKE_GPU** — 4×L4, Internet off, model dataset + wheelhouse + competition data attached. Two configurations,
   128 tokens. Look at `C0_frozen_q1 decode@8`: it must be ≈ 50 tok/s (anchor). If the anchor is off by more than
   ±40 % the measurement setup differs from Q1/PIV and the probe numbers cannot be compared to them.
3. **PROBE_GPU** — full ladder. Interactive session recommended (the per-config row prints as soon as each
   subprocess exits); `PROBE_CONFIGS = [...]` re-runs a subset. Stop the session after downloading the bundle.
4. Send `TP_Q27_PROBE_GPU_BUNDLE.zip` (+ the output-included notebook) — `REPORT.md`, `analysis/summary_table.csv`,
   `analysis/throughput_verdict.json`, per-config `logs/*.log` and `results/*.json`.

## Reading the numbers

- `agg_tps@n` is what fills the 12 h budget; `per_seq_tps@n` is what one agent lineage feels per turn.
  A good engine shows per-seq nearly flat while agg grows with n; the knee is where KV cache runs out.
- `all_reached_max_tokens=False` means a sequence stopped early (should not happen with `ignore_eos`) — treat that point as invalid.
- `prefill_warm_over_cold` ≫ 1 only when prefix caching is active; ≈ 1 otherwise.
- `fingerprint_vs_anchor.mean_prefix_agreement` = fraction of greedy tokens identical to the frozen engine. FP8 KV or
  different kernels legitimately drift; a value near 1.0 means numerics are effectively unchanged.
- `unsupported` status = the vLLM build rejected a kwarg (e.g. `mamba_cache_mode`), not a runtime failure.
